In [1]:
# Inicializar la base de datos

In [2]:
import sqlite3

def iniciar_base(): 
    # Abre el archivo de la base de datos. Si no existe, la crea de forma automática
    conn = sqlite3.connect('laboratorio.db')
    cursor = conn.cursor()

    # Habilitación de claves foráneas en sqlite
    cursor.execute("PRAGMA foreign_keys = ON;")

    # Creación de tabla de usuarios
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            usuario TEXT PRIMARY KEY,
            contrasena TEXT NOT NULL,
            rol TEXT NOT NULL CHECK(rol IN ('Recepcionista', 'Tecnologo')))''')

    # Creación de tabla de Paciente
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS pacientes (
            id_Paciente TEXT PRIMARY KEY,
            nombres TEXT NOT NULL,
            apellidos TEXT NOT NULL,
            rut TEXT UNIQUE NOT NULL,
            direccion TEXT,
            edad INTEGER,
            fecha_ingreso TEXT,
            correo TEXT)''')

    # Creación de tabla de exámenes
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS examenes (
            id_examen INTEGER PRIMARY KEY AUTOINCREMENT,
            id_Paciente TEXT,
            nombre_examen TEXT NOT NULL,
            resultado TEXT NOT NULL,
            fecha_examen TEXT,
            FOREIGN KEY (id_Paciente) REFERENCES pacientes(id_Paciente) ON DELETE CASCADE)''')
    

    # Defición de usuarios para ingresar en el login
    usuarios_demo = [
        ('admin_recep', '1234', 'Recepcionista'),
        ('admin_tec', '5678', 'Tecnologo')]
    cursor.executemany("INSERT OR IGNORE INTO usuarios VALUES (?, ?, ?)", usuarios_demo)

    # Guardar cambios y cerrar conexión
    conn.commit()
    conn.close()

iniciar_base()

OperationalError: database is locked

In [ ]:
# Enlace entre Login y Ventana Principal

In [ ]:
import sys
import sqlite3
import re
from PyQt5 import QtWidgets, uic
from PyQt5.QtWidgets import QMessageBox, QTableWidgetItem
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from PyQt5.QtCore import QDate
from PyQt5.QtGui import QIntValidator
from PyQt5.QtGui import QRegExpValidator
from PyQt5.QtCore import QRegExp
from PyQt5.QtWidgets import QInputDialog

# Ventana de Login que administra el acceso a la interfaz
class VentanaLogin(QtWidgets.QMainWindow):

    # Muestra la interfaz del login e inicializa el botón
    def __init__(self):
        super(VentanaLogin, self).__init__()
        uic.loadUi('ventana_principal.ui', self)
        self.boton_ingresar.clicked.connect(self.intentar_login)

        
    # Encargado de validar usuario y contraseñas ingresados, y permitir el acceso a la base 
    def intentar_login(self):
        usuario = self.txt_usuario.text()
        contrasena = self.txt_contrasena.text()

        # Conexión con la base de datos y búsqueda de la información de usuarios establecido
        conn = sqlite3.connect('laboratorio.db')
        cursor = conn.cursor()
        cursor.execute("SELECT rol FROM usuarios WHERE usuario = ? AND contrasena = ?", (usuario, contrasena))
        resultado = cursor.fetchone()
        conn.close()
        
        # Si el usuario existe en la base, se define el rol de recepcionista o tecnologo, y envía a la ventana secundaria
        # Si no existe, muestra un mensaje emergente de usuario no válido
        if resultado:
            rol_usuario = resultado[0]
            self.ventana_secundaria = MainWindowVentana(rol_usuario)
            self.ventana_secundaria.show()
            self.close()
        else:
            QMessageBox.warning(self, "Error de Autentificación", "Usuario o Contraseña Incorrectos")

# Ventana secundaria que permite el ingreso de pacientes, exámenes y estadisticas
class MainWindowVentana(QtWidgets.QMainWindow):

    # Carga la interfaz, guarda el rol con el que se inició sesión para definir las tabs que se bloquean o habilitan
    # Habilita botones e inicializa las funciones necesarias para mantener el funcionamiento
    def __init__(self, rol):
        super(MainWindowVentana, self).__init__()
        uic.loadUi('ventana_secundaria.ui', self)
        self.rol_usuario = rol
        print(f"Sesión iniciada con el rol: {self.rol_usuario}")
        self.restricciones_rol()
        self.txt_id.setReadOnly(True)
        self.txt_edad.setValidator(QIntValidator(0, 120))
        self.fecha_ingreso.setMaximumDate(QDate.currentDate())
        self.fecha_examen.setMaximumDate(QDate.currentDate())
        self.fecha_ingreso.setMinimumDate(QDate(2026, 6, 16))
        self.fecha_examen.setMinimumDate(QDate(2026, 6, 16))
        self.generar_id()
        self.boton_guardar_paciente.clicked.connect(self.guardar_paciente)
        self.boton_registrar_examen.clicked.connect(self.registrar_examen)
        self.boton_guardar_paciente_2.clicked.connect(self.buscar_paciente) 
        self.boton_guardar_paciente_3.clicked.connect(self.eliminar_registro)
        self.boton_modificar.clicked.connect(self.modificar_registro)
        self.boton_regresar1.clicked.connect(self.volver_login)
        self.boton_regresar2.clicked.connect(self.volver_login)
        self.actualizar_estadisticas_y_grafico()
        self.tableWidget.setColumnCount(4)
        self.tableWidget.setHorizontalHeaderLabels(
            ["ID Examen", "Nombre Examen", "Resultado", "Fecha"])
    
    # Aplica las reglas de usuario bloqueando o habilitando tabs dependiendo del usuario que inició sesión
    # Recepcionista: Ingreso, búsqueda de pacientes, y estadísticas
    # Tecnólogo: Ingreso de exámenes, búsqueda de pacientes y estadísticas
    def restricciones_rol(self):
        tabs = self.tab_widget
        if self.rol_usuario == "Recepcionista":
            tabs.setTabEnabled(1, False)
            tabs.setCurrentIndex(0)
            self.statusBar().showMessage("Conectado como: Recepcionista (Modo Lectura/Registro de Pacientes)")
        
        elif self.rol_usuario == "Tecnologo":
            tabs.setTabEnabled(0, False)
            tabs.setCurrentIndex(1)
            self.statusBar().showMessage("Conectado como: Tecnólogo Médico (Modo Gestión de Exámenes)")

    # Funcionamiento del botón regresar en la ventana secundaria para poder volver a la ventana principal y mensajes emergentes
    def volver_login(self):
        respuesta = QMessageBox.question(
            self,
            "Cerrar Sesión",
            "¿Desea volver a la pantalla de inicio de sesión?",
            QMessageBox.Yes | QMessageBox.No
        )
        if respuesta == QMessageBox.Yes:
            self.login = VentanaLogin()
            self.login.show()
            self.close()

    
    # Calcula de forma automática el ID del paciente dependiendo del último ID de la base de datos
    def generar_id(self):
        try:
            conn = sqlite3.connect('laboratorio.db')
            cursor = conn.cursor()
            cursor.execute("SELECT id_Paciente FROM pacientes")
            todos_los_ids = cursor.fetchall()
            conn.close()
            
            if todos_los_ids:
                numeros = [int(fila[0]) for fila in todos_los_ids if fila[0].isdigit()]
                if numeros:
                    siguiente_numero = max(numeros) + 1
                else:
                    siguiente_numero = 1
            else:
                siguiente_numero = 1

            id_formateado = str(siguiente_numero).zfill(4)
            self.txt_id.setText(id_formateado)
            
        except Exception as e:
            print(f"Error al generar ID automático: {e}")
            self.txt_id.setText("0001")
        
    # Verifica que los nombres ingresados contenga solo letras
    def validar_nombre(self, texto):
        patron = r'^[A-Za-zÁÉÍÓÚáéíóúÑñ ]+$'
        return re.match(patron, texto) is not None

    # Verifica que el correo ingresado tenga una estructura válida 
    def validar_correo_electronico(self, correo):
        patron = r'^[\w\.-]+@[\w\.-]+\.\w+$' 
        return re.match(patron, correo) is not None
        
    # Verifica el formato del rut, le quita los puntos o posibles espacios
    def validar_rut(self, rut): 
        rut = rut.replace(".", "").replace(" ", "").upper()
        patron = r'^\d{7,8}-[\dK]$'
        return re.match(patron, rut) is not None
        
        if not self.validar_rut(rut):
            QMessageBox.warning(
                self,
                "RUT INVÁLIDO",
                "Ingrese un rut con formato válido")
            return
        
    # Obtiene los textos ingresados en las casillas de texto, verifica y valida la información ingresada, envía mensajes emergentes en caso de no cumplir
    # Se conecta con la base de datos y guarda esa información en la tabla de pacientes
    # Actualiza los calculos y gráficos, y limpia la interfaz para un nuevo ingreso 
    # Detección de duplicados 
    def guardar_paciente(self):
        id_paciente = self.txt_id.text().strip()
        nombres = self.txt_nombres.text().strip()
        apellidos = self.txt_apellidos.text().strip()
        rut = self.txt_rut.text().strip()
        direccion = self.txt_direccion.text().strip()
        edad = self.txt_edad.text().strip()
        fecha = self.fecha_ingreso.date().toString("dd/MM/yyyy")
        correo = self.txt_correo.text().strip()

        if not id_paciente or not nombres or not apellidos or not rut or not correo:
            QMessageBox.warning(self, "Campos Vacíos", "Por favor, complete los campos obligatorios: ID, Nombres, Apellidos, RUT y Correo.")
            return

        if not self.validar_nombre(nombres):
            QMessageBox.warning(self, "Nombre Inválido", "El nombre solo puede contener letras")
            return

        if not self.validar_nombre(apellidos):
            QMessageBox.warning(self, "Apellido Inválido", "El apellido solo puede contener letras")
            return

        if not self.validar_rut(rut):
            QMessageBox.warning(self, "RUT Inválido", "El formato o el dígito verificador del RUT es incorrecto. Use el formato: 12345678-K")
            return

        if not self.validar_correo_electronico(correo):
            QMessageBox.warning(self, "Correo Inválido", "El formato del correo electrónico es incorrecto (ejemplo@correo.com).")
            return

        try:
            edad_int = int(edad) if edad else None

            if edad_int < 0 or edad_int > 120:
                QMessageBox.warning(
                    self,
                    "Edad Inválida",
                    "Ingrese un número entre 0 y 120")
                return
        except ValueError:
            QMessageBox.warning(
                self,
                "Edad Inválida",
                "La edad debe ser un número")
            return

        try:
            conn = sqlite3.connect('laboratorio.db')
            cursor = conn.cursor()
            cursor.execute('''
                INSERT INTO pacientes (id_Paciente, nombres, apellidos, rut, direccion, edad, fecha_ingreso, correo)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ''', (id_paciente, nombres, apellidos, rut, direccion, edad_int, fecha, correo))
            
            conn.commit()
            conn.close()
            QMessageBox.information(self, "Éxito", f"Paciente {nombres} {apellidos} registrado correctamente.")
            self.actualizar_estadisticas_y_grafico()
            
            self.txt_nombres.clear()
            self.txt_apellidos.clear()
            self.txt_rut.clear()
            self.txt_direccion.clear()
            self.txt_edad.clear()
            self.fecha_ingreso.setDate(QDate.currentDate())
            self.txt_correo.clear()
            self.generar_id()

        except sqlite3.IntegrityError as e:
            # Captura si el id_Paciente o el RUT ya existen en la base de datos
            conn.close()
            QMessageBox.critical(self, "Registro Duplicado", "Error: El ID de Paciente o el RUT ya se encuentran registrados en el sistema.")

    # Recibe, verifica y valida la información de exámenes vinculando a un paciente específico que ya debe estar registrado como paciente
    # Realiza la cuenta de exámenes registrados 
    # Si supera la verificación, el registro de exámen se guarda en la base
    # Realiza la limpieza de las casillas 
    def registrar_examen(self):
        try:
            id_asociado = self.txt_id_paciente_asociado.text().strip()
            tipo_examen = self.cb_tipo_examen.currentText()
            resultado = self.txt_resultado_examen.text().strip()
            fecha_examen = self.fecha_examen.date().toString("dd/MM/yyyy")
            
            if not id_asociado or not resultado or not fecha_examen:
                QMessageBox.warning(self, "Campos Vacíos", "Por favor, complete el ID Asociado, Resultado y Fecha del Examen.")
                return
                
            if id_asociado.isdigit():
                id_asociado = id_asociado.zfill(4)
                self.txt_id_paciente_asociado.setText(id_asociado)

            conn = sqlite3.connect('laboratorio.db')
            cursor = conn.cursor()

            # Conexión entre ambas tablas para comprobar si el ID del paciente existe ya en la tabla de paciente antes de guardar el exámen
            cursor.execute("SELECT nombres, apellidos FROM pacientes WHERE id_Paciente = ?", (id_asociado,))
            paciente_encontrado = cursor.fetchone()
            
            if not paciente_encontrado:
                conn.close()
                QMessageBox.critical(self, "Error de Clave Foránea", f"No se puede registrar el examen. El ID Paciente '{id_asociado}' NO existe en el sistema.")
                return
                
            cursor.execute("SELECT COUNT(*) FROM examenes WHERE id_Paciente = ?", (id_asociado,))
            cantidad_examenes = cursor.fetchone()[0]
            if cantidad_examenes >= 5:
                conn.close()
                QMessageBox.warning(self, "Límite Alcanzado", f"El paciente {paciente_encontrado[0]} ya tiene los 5 exámenes máximos permitidos.")
                return


            cursor.execute('''
                INSERT INTO examenes (id_Paciente, nombre_examen, resultado, fecha_examen)
                VALUES (?, ?, ?, ?)
            ''', (id_asociado, tipo_examen, resultado, fecha_examen))
            
            conn.commit()
            conn.close()
            QMessageBox.information(self, "Éxito", f"Examen '{tipo_examen}' registrado exitosamente para el paciente {paciente_encontrado[0]} {paciente_encontrado[1]}.")
            self.actualizar_estadisticas_y_grafico()
            
            self.txt_id_paciente_asociado.clear()
            self.txt_resultado_examen.clear()
            self.fecha_examen.setDate(QDate.currentDate())
            self.cb_tipo_examen.setCurrentIndex(0)
            
        except Exception as e:
            print(f"ERROR AL REGISTRAR EXAMEN: {e}")
            QMessageBox.critical(self, "Error Interno", f"No se pudo guardar el examen: {e}")

    # Busca exámenes en la base de datos cruzando la información de ambas tablas -> INNER JOIN
    # Se puede buscar por id, rut o nombre del paciente
    # Crea una tabla y llena la información
    def buscar_paciente(self):
        termino = self.txt_id_paciente_asociado_2.text().strip()
        termino_nombre = termino.upper()
        self.txt_nombre_encontrado.setText("PACIENTE: -")
        
        if not termino:
            QMessageBox.warning(self, "Atención", "Ingrese un ID, RUT o nombre para buscar.")
            return
                
        self.tableWidget.setColumnCount(4)
        self.tableWidget.setHorizontalHeaderLabels([
            "ID Examen",
            "Nombre Examen",
            "Resultado",
            "Fecha" 
        ])
        
        if termino.isdigit():
            termino_busca = termino.zfill(4)
        else:
            termino_busca = termino.replace(".", "").replace("-", "").upper()
        
        conn = sqlite3.connect('laboratorio.db')
        cursor = conn.cursor()
        # Cruce de tablas, haciendo coincidir la clave foránea de ID de paciente
        cursor.execute("""
        SELECT e.id_examen,
            e.nombre_examen,
            e.resultado,
            e.fecha_examen,
            p.rut,
            p.id_Paciente,
            p.nombres,
            p.apellidos
        FROM examenes e
        INNER JOIN pacientes p
            ON e.id_Paciente = p.id_Paciente
        """)
        
        resultados = cursor.fetchall()
        conn.close()

        if not resultados:
            QMessageBox.information(
                self,
                "Sin registros",
                "No existen exámenes registrados en la base de datos")
            return

        # Prepara la interfaz gráfica
        self.tableWidget.setRowCount(0)
        fila_tabla = 0
        nombre_mostrado = False
        
        # Algoritmo de filtrado
        for examen in resultados:
            id_examen = examen[0]
            nombre = examen[1]
            resultado = examen[2]
            fecha = examen[3]
            rut_bd = examen[4].replace(".", "").replace("-", "").upper()
            id_bd = str(examen[5]).zfill(4)
            nombre_bd = f"{examen[6]} {examen[7]}".upper()
            
            if (termino_busca == id_bd or termino_busca == rut_bd or termino_nombre in nombre_bd):
                if not nombre_mostrado:
                    nombre_completo = f"{examen[6]} {examen[7]}"
                    self.txt_nombre_encontrado.setText(nombre_completo)
                    nombre_mostrado = True
                    
                self.tableWidget.insertRow(fila_tabla)
                self.tableWidget.setItem(
                    fila_tabla, 0,
                    QTableWidgetItem(str(id_examen)))
                
                self.tableWidget.setItem(
                    fila_tabla, 1,
                    QTableWidgetItem(nombre))
                
                self.tableWidget.setItem(
                    fila_tabla, 2,
                    QTableWidgetItem(resultado))
                
                self.tableWidget.setItem(
                    fila_tabla, 3,
                    QTableWidgetItem(fecha))
                
                fila_tabla += 1
        
        if fila_tabla == 0:
            self.txt_nombre_encontrado.setText("PACIENTE NO ENCONTRADO")
            QMessageBox.information(
                self,
                "Búsqueda",
                "No se encontraron exámenes."
            )
        

    # Permite eliminar filas seleccionadas en la tabla 
    # Mensajes emergentes 
    # Eliminación de la información de la base de datos
    def eliminar_registro(self):
        if self.rol_usuario != "Tecnologo":
            QMessageBox.warning(
                self,
                "Acceso denegado",
                "Sólo un Tecnólogo puede eliminar exámenes."
            )
            return
        fila_seleccionada = self.tableWidget.currentRow()
        if fila_seleccionada < 0:
            QMessageBox.warning(self, "Atención", "Seleccione un examen de la tabla para eliminarlo.")
            return
            
        id_examen = self.tableWidget.item(fila_seleccionada, 0).text()
        
        confirmacion = QMessageBox.question(self, "Confirmar", "¿Está seguro de eliminar este examen?", 
                                            QMessageBox.Yes | QMessageBox.No)
        if confirmacion == QMessageBox.Yes:
            conn = sqlite3.connect('laboratorio.db')
            cursor = conn.cursor()
            cursor.execute("DELETE FROM examenes WHERE id_examen = ?", (id_examen,))
            conn.commit()
            conn.close()
            
            self.tableWidget.removeRow(fila_seleccionada)
            QMessageBox.information(self, "Éxito", "Registro eliminado correctamente.")
            self.actualizar_estadisticas_y_grafico()

    # Modifica los campos de una fila seleccionada, actualizando la información en la base de datos
    # Extrae el ID y pide la nueva información
    # Actualiza los valores y gráficas 
    def modificar_registro(self):
        if self.rol_usuario != "Tecnologo":
            QMessageBox.warning(
                self,
                "Acceso denegado",
                "Sólo un Tecnólogo puede modificar exámenes."
            )
            return
            
        fila = self.tableWidget.currentRow()
        if fila < 0:
            QMessageBox.warning(
                self,
                "Atención",
                "Seleccione un exámen de la tabla para modificar")
            return
            
        id_examen = self.tableWidget.item(fila, 0).text()
        tipo_actual = self.tableWidget.item(fila, 1).text()
        resultado_actual = self.tableWidget.item(fila, 2).text()
        fecha_actual = self.tableWidget.item(fila, 3).text()

        nuevo_tipo, ok = QInputDialog.getText(
            self,
            "Modificar Tipo de Examen",
            "Ingrese el nuevo tipo de examen: HEMOGRAMA COMPLETO, PERFIL BIOQUÍMICO, PERFIL LÍPIDICO, UROCULTIVO, PCR INMUNOLÓGICO",
            text=tipo_actual
        )
        if not ok or not nuevo_tipo.strip():
            return
        
        nuevo_resultado, ok = QInputDialog.getText(
            self,
            "Modificar Resultado",
            "Nuevo resultado:",
            text=resultado_actual
        )
        
        if not ok or not nuevo_resultado.strip():
            return

        nueva_fecha, ok = QInputDialog.getText(
            self,
            "Modificar Fecha",
            "Nueva fecha (dd/MM/yyyy):",
            text=fecha_actual
        )
        
        if not ok or not nueva_fecha.strip():
            return

        
        try:
            conn = sqlite3.connect('laboratorio.db')
            cursor = conn.cursor()
            cursor.execute("""
            UPDATE examenes
            SET nombre_examen = ?,
            resultado = ?,
            fecha_examen = ?
            WHERE id_examen = ?
        """, (
            nuevo_tipo,
            nuevo_resultado,
            nueva_fecha,
            id_examen
        ))

            conn.commit()
            conn.close()
            
            self.tableWidget.setItem(
                fila, 1,
                QTableWidgetItem(nuevo_tipo)
            )
            
            self.tableWidget.setItem(
                fila,
                2,
                QTableWidgetItem(nuevo_resultado.strip())
            )
            self.tableWidget.setItem(
                fila, 3,
                QTableWidgetItem(nueva_fecha)
            )
            
            QMessageBox.information(
                self,
                "Éxito",
                "Examen modificado correctamente."
            )
            
            self.actualizar_estadisticas_y_grafico()
        
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error",
                f"No se pudo modificar el examen:\n{e}"
            )
            
    # Realiza los calculos y gráficos con la información de la base de datos
    def actualizar_estadisticas_y_grafico(self):
        try:
            conn = sqlite3.connect('laboratorio.db')
            cursor = conn.cursor()

            # Cuenta la cantidad total de filas en la tabla de pacientes 
            cursor.execute("SELECT COUNT(*) FROM pacientes")
            total_pacientes = cursor.fetchone()[0]
            self.label_13.setText(f"TOTAL PACIENTES REGISTRADOS: {total_pacientes}")

            # Realiza el calculo del promedio de edades 
            cursor.execute("SELECT AVG(edad) FROM pacientes WHERE edad IS NOT NULL")
            promedio_edad = cursor.fetchone()[0]
            promedio_edad = round(promedio_edad, 1) if promedio_edad else 0
            self.label_14.setText(f"PROMEDIO DE EDAD: {promedio_edad} AÑOS")

            # Agrupa los exámenes por nombre
            cursor.execute("SELECT nombre_examen, COUNT(*) FROM examenes GROUP BY nombre_examen")
            datos_grafico = cursor.fetchall()
            conn.close()

            # Limpieza de memoria para evitar la superposición de valores en el gráfico
            if hasattr(self, 'canvas'):
                self.layout_grafico.removeWidget(self.canvas)
                self.canvas.deleteLater()

            # Construcción del gráfico 
            plt.ioff()
            fig, ax = plt.subplots(figsize=(4, 3))
            fig.patch.set_facecolor('#f0f0f0') 
            ax.set_facecolor('#f0f0f0')
            
            if datos_grafico:
                tipos = [row[0] for row in datos_grafico]
                cantidades = [row[1] for row in datos_grafico]
                ax.bar(tipos, cantidades, color='#00adb5', edgecolor='black', width=0.4)
                ax.set_title("Cantidad de Exámenes por Tipo", fontsize=9, fontweight='bold')
                from matplotlib.ticker import MaxNLocator
                ax.yaxis.set_major_locator(MaxNLocator(integer=True))
                plt.xticks(rotation=10, ha='right')
            else:
                ax.text(0.5, 0.5, "Sin exámenes en la BD", color='gray', ha='center', va='center')
                ax.axis('off')
                
            plt.tight_layout()
            
            self.canvas = FigureCanvas(fig)
            if not hasattr(self, 'layout_grafico') or self.layout_grafico is None:
                self.layout_grafico = QtWidgets.QVBoxLayout(self.widget)
                self.layout_grafico.setContentsMargins(0, 0, 0, 0)
            
            self.layout_grafico.addWidget(self.canvas)
            self.canvas.show()
            self.canvas.draw()
            
        except Exception as e:
            pass

# Inicia la interfaz
# Limpieza de la memoria 
# Finaliza el proceso 
if __name__ == '__main__':
    app = QtWidgets.QApplication.instance()
    if not app:
        app = QtWidgets.QApplication(sys.argv)
        
    login_app = VentanaLogin()
    login_app.show()
    code = app.exec_()
    plt.close('all')
    
    sys.exit(code)

Sesión iniciada con el rol: Recepcionista
